# Free Colab training — SIMULATED threat detection
Select Runtime → Change runtime type → T4 GPU if available. Otherwise set DEVICE='cpu'. Run cells in order. Save your notebook to Drive if desired, and download the model ZIP before the temporary runtime ends. No paid API is required. Read the Step 3 and Step 4 READMEs.


In [ ]:
import sys, subprocess, json, uuid, zipfile
from pathlib import Path
from datetime import datetime, timezone
REPO = Path('/content/network-threat-training')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Raimal-Raja/network-threat-detection-platform.git', str(REPO)], check=True)
print('Commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements-training.txt')], check=True)
print('Heavy imports run in fresh processes to avoid stale kernel packages.')


## Tests and verified ingestion
Tests must pass. The public source and processed hashes are verified. Unknown labels are excluded, not relabeled benign.


In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=REPO, check=True)
INGEST = """
from pathlib import Path
import certifi
from threat_platform.ingestion import download_source, ingest_source, verify_run
source = Path('data/raw/colab-scenario11.binetflow')
run = Path('data/processed/colab-run')
if not run.exists():
    source.parent.mkdir(parents=True, exist_ok=True)
    if not source.exists():
        download_source(source, ca_file=certifi.where())
    ingest_source(source, run)
print(verify_run(run))
"""
subprocess.run([sys.executable, '-c', INGEST], cwd=REPO, check=True)


## Train and inspect results
Use 'cuda' for GPU or 'cpu' when unavailable. Explicit CUDA must actually use GPU. Thresholds use validation only. Inspect test FPR and production_ready; high PR-AUC does not imply production approval.


In [ ]:
DEVICE = 'cuda'
RUN_NAME = 'colab-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
BUNDLE = REPO / 'artifacts' / RUN_NAME
subprocess.run([sys.executable, '-m', 'threat_platform.training', '--data', 'data/processed/colab-run', '--output', str(BUNDLE), '--device', DEVICE, '--target-fpr', '0.01'], cwd=REPO, check=True)
for filename in ['metrics.json', 'metadata.json', 'split-audit.json']:
    print(filename)
    print((BUNDLE / filename).read_text())


## Download and save to D drive
The ZIP contains the model bundle and guides. Save under D:\GitHub\network-threat-detection-platform\artifacts and unzip before inference. Saving the notebook alone does not preserve /content files. Do not commit data or model bundles to Git.


In [ ]:
ARCHIVE = Path('/content') / (RUN_NAME + '.zip')
with zipfile.ZipFile(ARCHIVE, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for file in sorted(BUNDLE.rglob('*')):
        if file.is_file():
            archive.write(file, Path('model') / file.relative_to(BUNDLE))
    for relative in ['docs/steps/03-features/README.md', 'docs/steps/04-training/README.md', 'docs/steps/04-training/verification.md']:
        archive.write(REPO / relative, relative)
print('Download:', ARCHIVE)
from google.colab import files
files.download(str(ARCHIVE))


## CPU inference and limits
Use predict_bundle with the extracted model folder and duration_us, packets, bytes and protocol inputs; see Step 4 for exact commands. Hashes and feature contracts are checked. XGBoost is exported even if validation recommends logistic regression. This is a development holdout within one short capture, not an independent operational evaluation.
